# 03 — Davis baseline comparison

## Goal

Compare completed Davis GCN, GAT, and GIN experiment records. Architecture ranking uses validation MSE only; test metrics are displayed after selection as final descriptive results.


## Setup


In [ ]:
from pathlib import Path
from importlib import import_module
import json
import subprocess
import sys


def find_project_root(start: Path = Path.cwd()) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "config.py").is_file() and (candidate / "src").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the project checkout.")


def require_api(module_name: str, attribute: str):
    module = import_module(module_name)
    if not hasattr(module, attribute):
        raise RuntimeError(
            f"Required v2 API {module_name}.{attribute} is unavailable. "
            "Update v2-refactor and rerun the tests before enabling this cell."
        )
    return getattr(module, attribute)


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

BRANCH = subprocess.check_output(
    ["git", "branch", "--show-current"], cwd=PROJECT_ROOT, text=True
).strip()
if BRANCH != "v2-refactor":
    raise RuntimeError(f"Expected v2-refactor, found {BRANCH!r}. v1-final is read-only.")

print(f"Project: {PROJECT_ROOT}")
print(f"Branch:  {BRANCH}")


In [ ]:
from config import EXPERIMENT_LOG_PATH

RUN_COMPARISON = False
MODELS = ("gcn", "gat", "gin")
SPLITS = ("random", "cold_drug", "cold_target")
SEEDS = (42, 123, 2026)
print("Experiment log:", EXPERIMENT_LOG_PATH)


## Steps


In [ ]:
if RUN_COMPARISON:
    import pandas as pd

    if not EXPERIMENT_LOG_PATH.exists():
        raise FileNotFoundError(EXPERIMENT_LOG_PATH)
    records = pd.read_csv(EXPERIMENT_LOG_PATH)
    required = {
        "dataset", "split", "model", "seed", "best_validation_mse",
        "test_mse", "test_rmse", "test_mae", "concordance_index",
        "pearson_correlation", "batch_size", "epochs",
        "learning_rate", "weight_decay", "split_fingerprint",
        "configuration_used",
    }
    missing = required.difference(records.columns)
    if missing:
        raise ValueError(f"Experiment log is missing columns: {sorted(missing)}")

    davis = records[
        records["dataset"].str.lower().eq("davis")
        & records["model"].str.lower().isin(MODELS)
        & records["split"].str.lower().isin(SPLITS)
    ].copy()
    if davis.empty:
        raise ValueError("No Davis baseline experiment records were found.")
    for column in ("dataset", "split", "model"):
        davis[column] = davis[column].astype(str).str.strip().str.lower()
    if davis["split_fingerprint"].isna().any() or (
        davis["split_fingerprint"].astype(str).str.strip().eq("").any()
    ):
        raise ValueError("Every Davis record needs a split_fingerprint.")

    # Keep the latest timestamp for duplicate reruns; row order is the
    # deterministic fallback for missing/unparseable timestamps.
    davis["_source_order"] = range(len(davis))
    if "timestamp" in davis.columns:
        davis["_timestamp_order"] = pd.to_datetime(
            davis["timestamp"], errors="coerce", utc=True
        )
        davis = davis.sort_values(
            ["_timestamp_order", "_source_order"],
            kind="stable",
            na_position="first",
        )
    davis = davis.drop_duplicates(
        ["dataset", "split", "seed", "model"], keep="last"
    )

    def parse_configuration(value):
        if isinstance(value, dict):
            return value
        if isinstance(value, str) and value.strip():
            try:
                parsed = json.loads(value)
            except json.JSONDecodeError as error:
                raise ValueError(
                    "configuration_used contains invalid JSON."
                ) from error
            if isinstance(parsed, dict):
                return parsed
        raise ValueError(
            "Every Davis record needs a JSON-object configuration_used."
        )

    configurations = davis["configuration_used"].map(parse_configuration)
    configuration_fields = (
        "max_protein_length",
        "early_stopping_patience",
        "minimum_improvement",
        "optimizer",
        "loss_function",
        "checkpoint_selection_metric",
        "test_evaluation_policy",
    )
    for field in configuration_fields:
        values = configurations.map(lambda configuration: configuration.get(field))
        missing_values = values.map(
            lambda value: value is None
            or (isinstance(value, str) and not value.strip())
        )
        if missing_values.any():
            raise ValueError(
                f"configuration_used must record {field!r} for every Davis run."
            )
        davis[f"configuration__{field}"] = values

    expected_models = set(MODELS)
    incomplete = {}
    for (split_name, seed), group in davis.groupby(["split", "seed"]):
        observed_models = set(group["model"])
        if observed_models != expected_models:
            incomplete[(split_name, int(seed))] = sorted(
                expected_models.difference(observed_models)
            )
    if incomplete:
        raise ValueError(
            f"Incomplete GCN/GAT/GIN Davis evidence: {incomplete}"
        )

    protocol_columns = [
        "batch_size", "epochs", "learning_rate", "weight_decay",
        "split_fingerprint",
        *[
            f"configuration__{field}"
            for field in configuration_fields
        ],
    ]
    protocol_mismatches = {}
    for (split_name, seed), group in davis.groupby(["split", "seed"]):
        differing = [
            column for column in protocol_columns
            if group[column].nunique(dropna=False) != 1
        ]
        if differing:
            protocol_mismatches[(split_name, int(seed))] = differing
    if protocol_mismatches:
        raise ValueError(
            "Davis baseline protocols differ within split/seed cells: "
            f"{protocol_mismatches}"
        )

    davis["validation_rank"] = davis.groupby(
        ["split", "seed"]
    )["best_validation_mse"].rank(method="min")
    summary = davis.sort_values(
        ["split", "seed", "best_validation_mse"]
    ).drop(columns=["_source_order", "_timestamp_order"], errors="ignore")
    print(summary.to_string(index=False, max_rows=60))
else:
    print("Comparison skipped: RUN_COMPARISON is False.")


## Checks

Duplicate reruns are reduced to the latest timestamp (or last row when unavailable). Every displayed split/seed must contain GCN, GAT, and GIN with identical split fingerprint, batch size, epoch budget, learning rate, weight decay, maximum protein length, early-stopping settings, optimiser, loss, checkpoint-selection metric, and test-evaluation policy. Do not use test RMSE, CI, Pearson, or any test-derived rank to choose the model.

## Next steps

Use notebook 04 to inspect cold-start integrity and generalisation gaps.
